# MODÈLE — `<source>` : `<table>` → landing (API, dlthub sur Databricks)

**Comment l'utiliser** (manuel 6, recette A) :
1. Importer ce notebook dans `ingestion/<source>/process/02_landing/01_loaders/` puis le renommer `<source>_<table>_l_loader`.
2. Chercher **`À ADAPTER`** (Ctrl + F) : 4 zones seulement (constantes, appels, extraction des lignes, contrôle d'erreur).
3. Les zones **`OPTIONNEL`** se gardent ou se suppriment (contrôle de quota API-Football, incrémental « déjà en landing »).
4. Lancer **Run all**, vérifier les sections 8 et 9, puis Commit & Push.

**Destination** : `/Volumes/<CATALOG>/landing/files/<source>/<table>/<aaaa>/<mm>/<jj>/<load_id>.<file_id>.parquet` (date du **dépôt**).

## 1. Paramètres (widgets)

`CATALOG` et `SECRET_SCOPE` sont toujours présents. Les autres widgets décrivent **ce qu'on demande à l'API** : À ADAPTER.

In [0]:
dbutils.widgets.text("CATALOG", "dev", "Catalog cible")
dbutils.widgets.text("SECRET_SCOPE", "scope-data-dev", "Secret scope")   # inutilisé (API sans clé) mais lu en cellule 4
# À ADAPTER : widget métier (test : une ou plusieurs villes du géocodage ; vide = toutes)
dbutils.widgets.text("VILLES", "", "Villes (vide = toutes celles du géocodage)")
dbutils.widgets.text("LOT_MAX", "2000", "Nombre max d'appels par lancement")   # limite Open-Meteo : 5 000/heure, 10 000/jour

## 2. Installation de dlthub (version fixée)

In [0]:
%pip install "dlt[parquet]==1.31.0"

In [0]:
%restart_python

## 3. Import de dlthub + contrôle « fail fast »

In [0]:
import dlt

if not hasattr(dlt, "__version__") or dlt.__file__.startswith("/databricks/"):
    raise ImportError("Conflit : le module 'dlt' chargé n'est pas dlthub (voir manuel 6, pièges)")
print("✅ dlthub", dlt.__version__)

## 4. Configuration

Toutes les valeurs propres à la source sont regroupées ici. **Aucune clé dans le code** : elle vient du secret scope.

In [0]:
import os, time
os.environ["RUNTIME__DLTHUB_TELEMETRY"] = "false"   # pas d'envoi de statistiques à dlthub
from dlt.sources.helpers import requests            # requests avec relances automatiques

def liste_entiers(texte: str) -> list[int]:
    """'61, 39,78' -> [61, 39, 78]"""
    return [int(x) for x in texte.split(",") if x.strip()]

CATALOG      = dbutils.widgets.get("CATALOG").strip()
SECRET_SCOPE = dbutils.widgets.get("SECRET_SCOPE").strip()

# ---------- À ADAPTER : identité de la source et de la table ----------
SOURCE   = "open_meteo"
TABLE    = "meteo_journaliere"
BASE_URL = "https://archive-api.open-meteo.com"
ENDPOINT = "/v1/archive"
SECRET_KEY_NAME = None
HEADER_NAME     = None
PAUSE_SECONDES  = 0.2      # limite Open-Meteo : 600 appels/minute -> 0,1 s suffit
TYPES_IMPOSES   = {}
# Variables journalières (8, donc sous le seuil de 10 qui ferait compter l'appel double)
VARIABLES_JOUR = "weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max"
# ----------------------------------------------------------------------

HEADERS = {}
if SECRET_KEY_NAME:
    HEADERS[HEADER_NAME] = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY_NAME)

LANDING = f"/Volumes/{CATALOG}/landing/files"
print(f"Catalog={CATALOG} | Source={SOURCE} | Table={TABLE} | Landing={LANDING}/{SOURCE}/{TABLE}/")

## 5. Liste des appels à faire

Un **appel** = un dictionnaire de paramètres envoyé à l'API. À ADAPTER selon la source.

on appelle ici toutes les villes qui sont présente dans api_football_fixture

In [0]:
VILLES_SAISIES = [v.strip() for v in dbutils.widgets.get("VILLES").split(",") if v.strip()]
filtre = ""
if VILLES_SAISIES:                                                     # mode test
    liste = ", ".join("'" + v.replace("'", "''") + "'" for v in VILLES_SAISIES)
    filtre = f"AND g._param_name IN ({liste})"

couples = spark.sql(f"""
    SELECT DISTINCT g.latitude, g.longitude, to_date(f.fixture__date) AS jour
    FROM {CATALOG}.bronze.api_football_fixtures f
    JOIN {CATALOG}.bronze.open_meteo_geocodage g
      ON f.fixture__venue__city = g._param_name                        -- ville du stade -> coordonnées
    WHERE to_date(f.fixture__date) <= date_sub(current_date(), 6)      -- archive : données dispo après ~5 jours
    {filtre}
""").collect()

APPELS = [{"latitude": r["latitude"], "longitude": r["longitude"],
           "start_date": str(r["jour"]), "end_date": str(r["jour"]),  # 1 seul jour -> compte pour 1 appel
           "daily": VARIABLES_JOUR, "timezone": "auto"}
          for r in couples]
print(f"{len(APPELS)} appel(s) prévu(s) : {APPELS[:2]}")

## 5 bis. OPTIONNEL — Incrémental : ne pas rappeler ce qui est déjà complet en landing

Exemple API-Football : un couple championnat × saison est complet si tous ses matchs ont un statut final. À adapter (colonnes, règle) ou à **supprimer**.

In [0]:

# Un couple coordonnées × jour déjà en landing n'est pas rappelé (la météo passée ne change pas)

CHEMIN_TABLE = f"{LANDING}/{SOURCE}/{TABLE}"

def absent(erreur: Exception) -> bool:
    return "PATH_NOT_FOUND" in str(erreur)

deja_charges = set()
try:
    deja = (spark.read.parquet(f"{CHEMIN_TABLE}/*/*/*/*.parquet")
            .select("_param_latitude", "_param_longitude", "_param_start_date").distinct().collect())
    deja_charges = {(r["_param_latitude"], r["_param_longitude"], r["_param_start_date"]) for r in deja}
except Exception as e:
    if not absent(e):
        raise
    print(f"ℹ️ Aucun fichier en landing pour {TABLE} (1er lancement ou dossier vidé)")

APPELS = [a for a in APPELS if (a["latitude"], a["longitude"], a["start_date"]) not in deja_charges]

print(f"Déjà en landing : {len(deja_charges)} | Appels restants : {len(APPELS)}")

LOT_MAX = int(dbutils.widgets.get("LOT_MAX"))
if len(APPELS) > LOT_MAX:                                   # on découpe : le reste au prochain lancement
    print(f"✂️ Lot limité à {LOT_MAX} appels ; {len(APPELS) - LOT_MAX} restants pour les lancements suivants")
    APPELS = APPELS[:LOT_MAX]
if len(APPELS) > 9000:                                                  # limite gratuite : 10 000 / jour
    raise RuntimeError(f"{len(APPELS)} appels : au-dessus de la limite journalière Open-Meteo. Arrêt.")
if not APPELS:
    dbutils.notebook.exit("Rien à charger : tous les jours de match sont déjà en landing.")

## 7. La ressource dlthub

Une fonction qui **produit** des lignes (`yield`). Chaque ligne reçoit aussi les paramètres de son appel (colonnes `_param_<nom>`), pour savoir d'où elle vient.

In [0]:
def extraire_lignes(reponse_json):
    # Open-Meteo archive : "daily" = colonnes en tableaux {"time": [...], "precipitation_sum": [...]} -> 1 ligne par jour
    jour = reponse_json.get("daily", {})
    lignes = [dict(zip(jour.keys(), valeurs)) for valeurs in zip(*jour.values())]
    for ligne in lignes:                                    # point de grille météo réellement utilisé (utile en silver)
        ligne["grille_latitude"]  = reponse_json.get("latitude")
        ligne["grille_longitude"] = reponse_json.get("longitude")
        ligne["grille_altitude"]  = reponse_json.get("elevation")
        ligne["fuseau_horaire"]   = reponse_json.get("timezone")
    return lignes

def verifier_erreurs(reponse_json, params):
    # Open-Meteo signale une erreur par "error": true + "reason"
    if reponse_json.get("error"):
        raise RuntimeError(f"Erreur API ({params}) : {reponse_json.get('reason')}")

@dlt.resource(name=TABLE, write_disposition="append", columns=TYPES_IMPOSES)   # types imposés (vide pour l'instant)
def ressource(appels: list[dict]):
    for i, params in enumerate(appels):
        if i > 0:
            time.sleep(PAUSE_SECONDES)                     # pause entre deux appels
        reponse_json = requests.get(f"{BASE_URL}{ENDPOINT}", headers=HEADERS, params=params).json()
        verifier_erreurs(reponse_json, params)
        lignes = extraire_lignes(reponse_json)
        print(f"📥 {params} : {len(lignes)} ligne(s)")
        yield [{**ligne, **{f"_param_{k}": v for k, v in params.items()}} for ligne in lignes]

## 8. Pipeline et chargement

`layout` = convention figée `<table>/<aaaa>/<mm>/<jj>/` (dlthub impose `{table_name}` avant la date).

In [0]:
pipeline = dlt.pipeline(
    pipeline_name=f"{SOURCE}_{TABLE}",
    destination=dlt.destinations.filesystem(
        bucket_url=f"file://{LANDING}",
        layout="{table_name}/{YYYY}/{MM}/{DD}/{load_id}.{file_id}.{ext}",
    ),
    dataset_name=SOURCE,
)
info = pipeline.run(ressource(APPELS), loader_file_format="parquet")
print(info)
info.raise_on_failed_jobs()

## 9. Vérifications : fichiers du jour et contenu

In [0]:
from datetime import datetime, timezone
dossier_jour = f"{LANDING}/{SOURCE}/{TABLE}/{datetime.now(timezone.utc):%Y/%m/%d}"
for f in sorted(os.listdir(dossier_jour)):
    print("📄", f)

df = spark.read.parquet(dossier_jour)
print("Colonnes :", len(df.columns), "| Lignes (dépôts du jour) :", df.count())
display(df.limit(20))